# Chapter 52 — What Is RAG?
Build RAG manually with embeddings, retrieval, and Groq
RAG = Retrieval + Augmented context + Generation. This standalone notebook builds a small RAG pipeline: create a knowledge base, embed passages, retrieve relevant passages with FAISS, pass them to a Groq-hosted model through the OpenAI-compatible API, and inspect the generated answer and source evidence.

# 1. Install dependencies

In [ ]:
%pip install -q sentence-transformers faiss-cpu numpy pandas openai

# 2. Imports 

In [1]:
import os
from getpass import getpass
import numpy as np
import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer
from openai import OpenAI

/Users/vimal/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


# 3. Create a small knowledge base

In [2]:
documents = [
    {"id":"KB001", "topic":"cross_border", "text":"A bank may review international transfers that differ from a customer's expected activity, including unusual frequency, amount, destination, or payment purpose."},
    {"id":"KB002", "topic":"monitoring", "text":"Transaction monitoring systems use rules or analytical models to identify transactions that may require review. An alert is a signal for investigation, not proof of wrongdoing."},
    {"id":"KB003", "topic":"investigation", "text":"An investigator should review alert facts, the customer profile, transaction history, and available supporting evidence. The decision and rationale should be documented."},
    {"id":"KB004", "topic":"alert_quality", "text":"A false positive is an alert reviewed and found not to represent suspicious activity. Clear documentation helps explain why an alert was closed."},
    {"id":"KB005", "topic":"sanctions", "text":"Sanctions screening checks relevant customers and payment parties against applicable sanctions lists. Potential matches require assessment under approved procedures."},
    {"id":"KB006", "topic":"kyc", "text":"Know Your Customer information helps establish customer identity, business activity, expected transaction behavior, and context for assessing unusual activity."},
    {"id":"KB007", "topic":"escalation", "text":"When evidence or risk indicators remain unresolved, an investigator should follow the institution's escalation procedures rather than make unsupported assumptions."},
    {"id":"KB008", "topic":"data_quality", "text":"Before relying on transaction data, analysts should check key fields, valid formats, and whether records are linked to the correct customer and payment."}
]

In [4]:
knowledge_base = pd.DataFrame(documents)
knowledge_base

,id,topic,text
0,KB001,cross_border,A bank may review international transfers that...
1,KB002,monitoring,Transaction monitoring systems use rules or an...
2,KB003,investigation,"An investigator should review alert facts, the..."
3,KB004,alert_quality,A false positive is an alert reviewed and foun...
4,KB005,sanctions,Sanctions screening checks relevant customers ...
5,KB006,kyc,Know Your Customer information helps establish...
6,KB007,escalation,When evidence or risk indicators remain unreso...
7,KB008,data_quality,"Before relying on transaction data, analysts s..."


# 4. Embed the documents
The embedding model converts each passage into a numerical vector. Here each vector has 384 dimensions. We normalize vectors so inner-product ranking corresponds to cosine similarity.

In [8]:
embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
document_texts = [d["text"] for d in documents]

In [9]:
document_texts

["A bank may review international transfers that differ from a customer's expected activity, including unusual frequency, amount, destination, or payment purpose.",
 'Transaction monitoring systems use rules or analytical models to identify transactions that may require review. An alert is a signal for investigation, not proof of wrongdoing.',
 'An investigator should review alert facts, the customer profile, transaction history, and available supporting evidence. The decision and rationale should be documented.',
 'A false positive is an alert reviewed and found not to represent suspicious activity. Clear documentation helps explain why an alert was closed.',
 'Sanctions screening checks relevant customers and payment parties against applicable sanctions lists. Potential matches require assessment under approved procedures.',
 'Know Your Customer information helps establish customer identity, business activity, expected transaction behavior, and context for assessing unusual activity.

In [10]:
document_embeddings = embedding_model.encode(
    document_texts, convert_to_numpy=True,
    normalize_embeddings=True, show_progress_bar=True
).astype("float32")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [11]:
print("Embedding matrix shape:", document_embeddings.shape)
print("Data type:", document_embeddings.dtype)
assert document_embeddings.shape[0] == len(documents)
assert document_embeddings.ndim == 2

Embedding matrix shape: (8, 384)
Data type: float32


# 5. Build the FAISS index

In [13]:
vector_index = faiss.IndexFlatIP(document_embeddings.shape[1])

In [14]:
vector_index

<faiss.swigfaiss.IndexFlatIP; proxy of <Swig Object of type 'faiss::IndexFlatIP *' at 0x14b3a2ba0> >

In [15]:
vector_index.add(document_embeddings)

In [16]:
vector_index.ntotal

8

In [17]:
assert vector_index.ntotal == len(documents)


# 6. Write the retrieval function

In [25]:
def retrieve_documents(query: str, k: int = 3) -> list[dict]:
    if not isinstance(query, str) or not query.strip():
        raise ValueError("query must be a non-empty string")
    if not isinstance(k, int) or not 1 <= k <= len(documents):
        raise ValueError(f"k must be between 1 and {len(documents)}")

    query_embedding = embedding_model.encode(
        [query], convert_to_numpy=True, normalize_embeddings=True
    ).astype("float32")

    scores, positions = vector_index.search(query_embedding, k)
    retrieved = []
    for score, position in zip(scores[0], positions[0]):
        if position == -1:
            continue
        record = documents[int(position)]
        retrieved.append({
            "id": record["id"],
            "topic": record["topic"],
            "text": record["text"],
            "similarity": float(score)
        })
    return retrieved

# 7. Inspect retrieval before generation

In [26]:
question = "What should an analyst review before closing an unusual international payment alert?"
retrieved = retrieve_documents(question, k=3)

print("QUESTION:", question)
for rank, item in enumerate(retrieved, start=1):
    print(f"\n{rank}. {item['id']} | {item['topic']} | similarity={item['similarity']:.4f}")
    print(item["text"])

QUESTION: What should an analyst review before closing an unusual international payment alert?

1. KB002 | monitoring | similarity=0.5462
Transaction monitoring systems use rules or analytical models to identify transactions that may require review. An alert is a signal for investigation, not proof of wrongdoing.

2. KB001 | cross_border | similarity=0.4765
A bank may review international transfers that differ from a customer's expected activity, including unusual frequency, amount, destination, or payment purpose.

3. KB004 | alert_quality | similarity=0.4755
A false positive is an alert reviewed and found not to represent suspicious activity. Clear documentation helps explain why an alert was closed.


# 8. Configure Groq securely

In [27]:
api_key = os.getenv("GROQ_API_KEY")
if not api_key:
    api_key = getpass("Enter your Groq API key (input hidden): ").strip()
if not api_key:
    raise ValueError("Set GROQ_API_KEY or enter a valid key.")

client = OpenAI(
    api_key=api_key,
    base_url="https://api.groq.com/openai/v1"
)
GROQ_MODEL = "openai/gpt-oss-20b"
print("Groq client configured. Model:", GROQ_MODEL)
# Never print the API key.

Groq client configured. Model: openai/gpt-oss-20b


# 9. Build the augmented prompt
The prompt combines the question with retrieved passages and their source IDs. Instructions ask Gemini not to invent missing facts and to cite source IDs. Prompting reduces risk but cannot guarantee a hallucination-free answer.

In [33]:
def build_rag_prompt(question: str, retrieved: list[dict]) -> str:
    context = "\n\n".join(
        f"[Source ID: {item['id']} | Topic: {item['topic']}]\n{item['text']}"
        for item in retrieved
    )
    return f"""You are a careful assistant answering from a supplied knowledge base.
Rules:
1. Use only the context below for factual claims.
2. Do not invent facts, thresholds, or procedures.
3. If the context is insufficient, say what information is missing.
4. Cite relevant source IDs in square brackets, e.g. [KB003].
5. A risk indicator is not proof of wrongdoing.

CONTEXT:
{context}

QUESTION:
{question}

Answer clearly and concisely, citing the supplied source IDs."""

In [34]:
def generate_answer(prompt: str) -> str:
    response = client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.2
    )
    answer = response.choices[0].message.content
    if not answer:
        raise RuntimeError("Groq returned an empty response.")
    return answer

rag_prompt = build_rag_prompt(question, retrieved)
answer = generate_answer(rag_prompt)
print("QUESTION:", question)
print("\nRAG ANSWER:\n")
print(answer)

QUESTION: What should an analyst review before closing an unusual international payment alert?

RAG ANSWER:

An analyst should:

1. Examine the transaction details and compare them to the customer’s expected activity – looking for unusual frequency, amount, destination, or payment purpose.  
2. Verify that the alert is a false positive by confirming no suspicious activity exists.  
3. Document the review and the reasons for closing the alert.  

These steps align with the monitoring framework and alert‑quality guidelines. [KB001] [KB004] [KB002]



# 11. Combine retrieval and generation into one function

In [35]:
def answer_with_rag(question: str, k: int = 3) -> dict:
    passages = retrieve_documents(question, k=k)
    prompt = build_rag_prompt(question, passages)
    answer = generate_answer(prompt)
    return {"question": question, "answer": answer, "sources": passages}

result = answer_with_rag(
    "Why should an investigator document the rationale for closing an alert?",
    k=3
)
print("QUESTION:", result["question"])
print("\nANSWER:\n", result["answer"])
print("\nSOURCES:")
for source in result["sources"]:
    print(f"- {source['id']} ({source['topic']}, similarity={source['similarity']:.3f})")
    print(" ", source["text"])

QUESTION: Why should an investigator document the rationale for closing an alert?

ANSWER:
 An investigator should document the rationale for closing an alert to provide clear, auditable evidence that the alert was reviewed, the relevant facts (customer profile, transaction history, supporting evidence) were considered, and the decision was justified as a false positive or non‑suspicious activity. This documentation explains why the alert was closed and supports compliance and future reference. [KB003] [KB004]

SOURCES:
- KB003 (investigation, similarity=0.643)
  An investigator should review alert facts, the customer profile, transaction history, and available supporting evidence. The decision and rationale should be documented.
- KB004 (alert_quality, similarity=0.616)
  A false positive is an alert reviewed and found not to represent suspicious activity. Clear documentation helps explain why an alert was closed.
- KB002 (monitoring, similarity=0.582)
  Transaction monitoring systems

# 12. Experiment with Top-K

In [36]:
experiment_question = "How should an analyst assess a potentially unusual cross-border payment?"
for k_value in [1, 3, 5]:
    print(f"\nTOP-K = {k_value}")
    for passage in retrieve_documents(experiment_question, k=k_value):
        print(f"- {passage['id']} | {passage['topic']} | {passage['similarity']:.3f}")
        print(" ", passage["text"])


TOP-K = 1
- KB001 | cross_border | 0.519
  A bank may review international transfers that differ from a customer's expected activity, including unusual frequency, amount, destination, or payment purpose.

TOP-K = 3
- KB001 | cross_border | 0.519
  A bank may review international transfers that differ from a customer's expected activity, including unusual frequency, amount, destination, or payment purpose.
- KB005 | sanctions | 0.438
  Sanctions screening checks relevant customers and payment parties against applicable sanctions lists. Potential matches require assessment under approved procedures.
- KB008 | data_quality | 0.407
  Before relying on transaction data, analysts should check key fields, valid formats, and whether records are linked to the correct customer and payment.

TOP-K = 5
- KB001 | cross_border | 0.519
  A bank may review international transfers that differ from a customer's expected activity, including unusual frequency, amount, destination, or payment purpose.
- K

# 13. Ask something the knowledge base does not answer

In [37]:
unknown_question = "What exact monetary threshold must trigger an alert in every country?"
unknown_result = answer_with_rag(unknown_question, k=3)
print("QUESTION:", unknown_result["question"])
print("\nANSWER:\n", unknown_result["answer"])
print("\nSOURCES PROVIDED:")
for source in unknown_result["sources"]:
    print(source["id"], "—", source["text"])

QUESTION: What exact monetary threshold must trigger an alert in every country?

ANSWER:
 The supplied sources do not specify a single monetary threshold that must trigger an alert in every country. None of the references (KB002, KB004, KB001) provide a concrete dollar amount or rule that applies universally. Therefore, I cannot provide an exact threshold based on the available context.

SOURCES PROVIDED:
KB002 — Transaction monitoring systems use rules or analytical models to identify transactions that may require review. An alert is a signal for investigation, not proof of wrongdoing.
KB004 — A false positive is an alert reviewed and found not to represent suspicious activity. Clear documentation helps explain why an alert was closed.
KB001 — A bank may review international transfers that differ from a customer's expected activity, including unusual frequency, amount, destination, or payment purpose.
